# Hoja de trabajo 2 — Transformers y mecanismos de atención
**CC3092 Deep Learning y Sistemas Inteligentes** · Fernando Andree Hernández Martínez (23645)

Contenido del notebook:
1. Configuración y semillas
2. Ejercicio a mano de *scaled dot-product attention* y su verificación numérica
3. Verificaciones contra `F.scaled_dot_product_attention`, `nn.MultiheadAttention` y máscaras causales
4. Escalamiento por $\sqrt{d_k}$: varianza de $q\cdot k$, saturación de la softmax y gradientes
5. Costo computacional de la self-attention respecto a $n$
6. Mapas de atención en BERT y GPT-2 (patrones por cabeza)
7. Correferencia de «it» (*tired* vs *wide*)
8. GPT-2: matriz triangular inferior y *attention sinks*
9. Entropía de la atención por capa
10. (Opcional) bertviz

In [ ]:
# Configuración: semillas fijas para reproducibilidad y rutas del proyecto
import sys, math, time, random
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
FIG = ROOT / "figures"
FIG.mkdir(exist_ok=True)

from attention import (scaled_dot_product_attention, causal_mask, multi_head_attention,
                       get_attentions, attention_entropy, head_patterns, plot_heads)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.set_printoptions(precision=4, sci_mode=False)
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 200, "savefig.bbox": "tight"})
print("torch", torch.__version__)

## 2. Ejercicio a mano

Tres tokens con $d_k = d_v = 2$ y, para que las cuentas sean manejables, $W_Q=W_K=W_V=I$, de modo que $Q=K=V=X$:

$$X=\begin{pmatrix}1&0\\0&1\\1&1\end{pmatrix}$$

**Paso 1 — puntajes** $S = QK^\top$:
$S=\begin{pmatrix}1&0&1\\0&1&1\\1&1&2\end{pmatrix}$

**Paso 2 — escalar** por $\sqrt{d_k}=\sqrt2\approx1.4142$:
$S/\sqrt2=\begin{pmatrix}0.7071&0&0.7071\\0&0.7071&0.7071\\0.7071&0.7071&1.4142\end{pmatrix}$

**Paso 3 — softmax por fila.** Fila 1: $e^{0.7071}=2.0281$, $e^0=1$ → suma $5.0562$ → $(0.4011,\,0.1978,\,0.4011)$.
Fila 2 es la permutación de la fila 1: $(0.1978,\,0.4011,\,0.4011)$.
Fila 3: $e^{0.7071}=2.0281$, $e^{1.4142}=4.1133$ → suma $8.1695$ → $(0.2483,\,0.2483,\,0.5035)$.

**Paso 4 — salida** $Z = AV$:
- $z_1 = 0.4011(1,0)+0.1978(0,1)+0.4011(1,1) = (0.8022,\,0.5989)$
- $z_2 = (0.5989,\,0.8022)$
- $z_3 = 0.2483(1,0)+0.2483(0,1)+0.5035(1,1) = (0.7517,\,0.7517)$

Interpretación: el token 3 ($[1,1]$) es el más parecido a todos, por eso recibe más peso en todas las filas; cada salida es un promedio ponderado de los valores.

In [ ]:
# Verificación numérica del ejercicio a mano
X = torch.tensor([[1., 0.], [0., 1.], [1., 1.]])
S = X @ X.T
S_scaled = S / math.sqrt(2)
A_hand = torch.softmax(S_scaled, dim=-1)
Z_hand = A_hand @ X
print("S =\n", S, "\nS/sqrt(2) =\n", S_scaled, "\nA =\n", A_hand, "\nZ = A V =\n", Z_hand)

Z_ref = F.scaled_dot_product_attention(X[None], X[None], X[None])[0]
A_expected = torch.tensor([[0.4011, 0.1978, 0.4011], [0.1978, 0.4011, 0.4011], [0.2483, 0.2483, 0.5035]])
Z_expected = torch.tensor([[0.8022, 0.5989], [0.5989, 0.8022], [0.7517, 0.7517]])
print("A coincide con lo calculado a mano:", torch.allclose(A_hand, A_expected, atol=1e-4))
print("Z coincide con lo calculado a mano:", torch.allclose(Z_hand, Z_expected, atol=1e-4))
print("Z coincide con F.scaled_dot_product_attention:", torch.allclose(Z_hand, Z_ref, atol=1e-6))

## 3. Verificaciones de la implementación

Se compara la implementación propia (`src/attention.py`) con las de PyTorch:
1. `scaled_dot_product_attention` propia vs `F.scaled_dot_product_attention` (sin máscara y con `is_causal=True`).
2. `multi_head_attention` propia vs `nn.MultiheadAttention` copiando sus pesos (`in_proj_weight`, `out_proj.weight`).
3. Propiedades: filas de la atención suman 1, la máscara causal produce una matriz triangular inferior, y `nn.Transformer.generate_square_subsequent_mask` equivale a nuestra máscara booleana.

In [ ]:
B, h, n, d = 2, 4, 7, 16
Q, K, V = torch.randn(B, h, n, d), torch.randn(B, h, n, d), torch.randn(B, h, n, d)

out_mine, w_mine = scaled_dot_product_attention(Q, K, V)
out_torch = F.scaled_dot_product_attention(Q, K, V)
out_mine_c, w_mine_c = scaled_dot_product_attention(Q, K, V, mask=causal_mask(n))
out_torch_c = F.scaled_dot_product_attention(Q, K, V, is_causal=True)

# nn.MultiheadAttention con los mismos pesos que nuestra implementación
d_model, heads = 32, 4
mha = nn.MultiheadAttention(d_model, heads, batch_first=True, bias=False).eval()
x = torch.randn(B, n, d_model)
W_q, W_k, W_v = mha.in_proj_weight.chunk(3)
ours, w_ours = multi_head_attention(x, W_q, W_k, W_v, mha.out_proj.weight, heads)
ref, w_ref = mha(x, x, x, need_weights=True, average_attn_weights=False)

# Máscara de PyTorch: float con -inf arriba de la diagonal y 0 en el resto
torch_mask = nn.Transformer.generate_square_subsequent_mask(n)

checks = {
    "SDPA propia == F.scaled_dot_product_attention": torch.allclose(out_mine, out_torch, atol=1e-5),
    "SDPA causal propia == F.sdpa(is_causal=True)": torch.allclose(out_mine_c, out_torch_c, atol=1e-5),
    "MHA propia == nn.MultiheadAttention (salida)": torch.allclose(ours, ref, atol=1e-5),
    "MHA propia == nn.MultiheadAttention (pesos)": torch.allclose(w_ours, w_ref, atol=1e-5),
    "Filas de atención suman 1": torch.allclose(w_mine.sum(-1), torch.ones(B, h, n)),
    "Atención causal es triangular inferior": bool((w_mine_c.triu(1) == 0).all()),
    "generate_square_subsequent_mask == ~causal_mask": torch.equal(torch.isinf(torch_mask), ~causal_mask(n)),
}
for k, v in checks.items():
    print(f"[{'OK' if v else 'FALLA'}] {k}")
print("Error máximo MHA:", (ours - ref).abs().max().item())

## 4. ¿Por qué dividir entre $\sqrt{d_k}$?

Si $q_i, k_i \sim \mathcal N(0,1)$ independientes, $q\cdot k=\sum_{i=1}^{d_k} q_ik_i$ es suma de $d_k$ términos con media 0 y varianza $E[q_i^2]E[k_i^2]=1$, así que $\operatorname{Var}(q\cdot k)=d_k$. Dividir entre $\sqrt{d_k}$ devuelve la varianza a 1. Se verifica empíricamente y se mide el efecto sobre la softmax (probabilidad máxima) y sobre la norma del gradiente de la softmax.

In [ ]:
dks = [4, 16, 64, 256, 1024]
n_tokens, trials = 32, 2000
rows = []
for dk in dks:
    q = torch.randn(trials, dk); k = torch.randn(trials, dk)
    var_raw = (q * k).sum(-1).var().item()
    var_scaled = ((q * k).sum(-1) / math.sqrt(dk)).var().item()
    res = {}
    for scaled in (False, True):
        Qs = torch.randn(200, 1, dk); Ks = torch.randn(200, n_tokens, dk)
        logits = (Qs @ Ks.transpose(-2, -1)).squeeze(1)
        if scaled:
            logits = logits / math.sqrt(dk)
        logits.requires_grad_(True)
        p = torch.softmax(logits, -1)
        # gradiente de la softmax respecto a sus logits con una señal aleatoria aguas arriba
        (p * torch.randn_like(p)).sum().backward()
        res[scaled] = (p.max(-1).values.mean().item(), logits.grad.norm(dim=-1).mean().item())
    rows.append((dk, var_raw, var_scaled, *res[False], *res[True]))

print(f"{'d_k':>5} {'Var(q·k)':>9} {'Var esc.':>9} {'max p s/esc':>11} {'|grad| s/esc':>12} {'max p esc':>10} {'|grad| esc':>10}")
for r in rows:
    print(f"{r[0]:>5} {r[1]:>9.1f} {r[2]:>9.3f} {r[3]:>11.3f} {r[4]:>12.4f} {r[5]:>10.3f} {r[6]:>10.4f}")

fig, axes = plt.subplots(1, 3, figsize=(12, 3.3))
dk_arr = np.array(dks)
axes[0].loglog(dk_arr, [r[1] for r in rows], "o-", label="sin escalar")
axes[0].loglog(dk_arr, [r[2] for r in rows], "s-", label=r"escalado $/\sqrt{d_k}$")
axes[0].loglog(dk_arr, dk_arr, "k--", lw=0.8, label="$d_k$ (teórico)")
axes[0].set(title=r"Varianza de $q\cdot k$", xlabel="$d_k$"); axes[0].legend(fontsize=8)
axes[1].semilogx(dk_arr, [r[3] for r in rows], "o-", label="sin escalar")
axes[1].semilogx(dk_arr, [r[5] for r in rows], "s-", label="escalado")
axes[1].axhline(1 / n_tokens, color="k", ls="--", lw=0.8, label="uniforme 1/n")
axes[1].set(title="Prob. máxima de la softmax (n=32)", xlabel="$d_k$"); axes[1].legend(fontsize=8)
axes[2].loglog(dk_arr, [r[4] for r in rows], "o-", label="sin escalar")
axes[2].loglog(dk_arr, [r[6] for r in rows], "s-", label="escalado")
axes[2].set(title="Norma del gradiente de la softmax", xlabel="$d_k$"); axes[2].legend(fontsize=8)
fig.tight_layout(); fig.savefig(FIG / "escalamiento.png"); plt.show()

**Lectura:** sin escalar, la varianza crece linealmente con $d_k$; los logits se separan tanto que la softmax se vuelve casi *one-hot* (prob. máxima → 1) y su jacobiano $\operatorname{diag}(p)-pp^\top$ tiende a cero, por lo que el gradiente se desvanece. Con el escalamiento la varianza se mantiene ≈1 y la distribución y el gradiente quedan estables para cualquier $d_k$.

## 5. Costo computacional respecto a $n$

Se mide tiempo y memoria de la matriz de atención para longitudes crecientes ($d=64$, una cabeza, CPU). La memoria de la matriz $n\times n$ en float32 es $4n^2$ bytes; el tiempo debería crecer como $O(n^2 d)$.

In [ ]:
def bench(fn, reps=5):
    fn()  # calentamiento
    t = time.perf_counter()
    for _ in range(reps):
        fn()
    return (time.perf_counter() - t) / reps

d = 64
ns = [256, 512, 1024, 2048, 4096, 8192]
n_threads = torch.get_num_threads()
torch.set_num_threads(1)  # un solo hilo: el tiempo refleja el número de operaciones, no el paralelismo
t_manual, t_fused, mem_mb = [], [], []
for n_ in ns:
    q = torch.randn(1, 1, n_, d); k = torch.randn(1, 1, n_, d); v = torch.randn(1, 1, n_, d)
    t_manual.append(bench(lambda: scaled_dot_product_attention(q, k, v)))
    t_fused.append(bench(lambda: F.scaled_dot_product_attention(q, k, v)))
    mem_mb.append(4 * n_ * n_ / 2**20)  # matriz de pesos n×n en float32
torch.set_num_threads(n_threads)

slope = np.polyfit(np.log(ns[2:]), np.log(t_manual[2:]), 1)[0]
print("Pendiente log-log del tiempo (implementación manual):", round(slope, 2))
for n_, a, b, m in zip(ns, t_manual, t_fused, mem_mb):
    print(f"n={n_:>5}  manual={a*1e3:8.2f} ms  F.sdpa={b*1e3:8.2f} ms  matriz n×n={m:8.1f} MB")

fig, axes = plt.subplots(1, 2, figsize=(9, 3.3))
axes[0].loglog(ns, np.array(t_manual) * 1e3, "o-", label="manual (materializa n×n)")
axes[0].loglog(ns, np.array(t_fused) * 1e3, "s-", label="F.scaled_dot_product_attention")
ref = np.array(ns, float) ** 2; axes[0].loglog(ns, ref / ref[-1] * t_manual[-1] * 1e3, "k--", lw=0.8, label="$O(n^2)$")
axes[0].set(title="Tiempo de self-attention", xlabel="n (tokens)", ylabel="ms"); axes[0].legend(fontsize=8)
n_big = np.array([1e3, 4e3, 16e3, 32e3, 100e3])
axes[1].loglog(ns, mem_mb, "o-", label="medido (n×n float32)")
axes[1].loglog(n_big, 4 * n_big**2 / 2**30 * 1024, "k--", lw=0.8, label="extrapolado")
axes[1].axhline(80 * 1024, color="r", ls=":", label="80 GB (GPU H100)")
axes[1].set(title="Memoria de la matriz de atención\n(1 cabeza, 1 capa)", xlabel="n (tokens)", ylabel="MB"); axes[1].legend(fontsize=8)
fig.tight_layout(); fig.savefig(FIG / "costo.png"); plt.show()
print(f"n=100,000 → {4*1e5**2/2**30:.1f} GB por cabeza y capa sólo para la matriz de atención")

**Lectura:** el tiempo crece con pendiente ≈2 en escala log-log para $n\ge1024$ (cuadrático; con $n$ pequeño domina el overhead) y la memoria de la matriz es exactamente $4n^2$ bytes: con $n=100{,}000$ serían ~37 GB *por cabeza y por capa*. La versión fusionada de PyTorch es más rápida porque no siempre materializa la matriz completa (kernels tipo FlashAttention / memory-efficient), pero el número de operaciones sigue siendo $O(n^2d)$.

## 6. Atención en modelos preentrenados

Se cargan BERT (`bert-base-uncased`, encoder bidireccional, 12 capas × 12 cabezas) y GPT-2 (decoder causal, 12 × 12) con `output_attentions=True` y `attn_implementation="eager"` para que devuelvan los pesos de atención. `outputs.attentions` es una tupla con una entrada por capa de forma `(batch, cabezas, n, n)`; aquí se apilan en un tensor `(capas, cabezas, n, n)`.

In [ ]:
from transformers import AutoTokenizer, AutoModel
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error()

tok_bert = AutoTokenizer.from_pretrained("bert-base-uncased")
bert = AutoModel.from_pretrained("bert-base-uncased", output_attentions=True, attn_implementation="eager").eval()
tok_gpt2 = AutoTokenizer.from_pretrained("gpt2")
gpt2 = AutoModel.from_pretrained("gpt2", output_attentions=True, attn_implementation="eager").eval()

SENT = "The quick brown fox jumps over the lazy dog, and the dog sleeps."
tok_b, A_bert = get_attentions(bert, tok_bert, SENT)
tok_g, A_gpt2 = get_attentions(gpt2, tok_gpt2, SENT)
print("BERT:", A_bert.shape, tok_b)
print("GPT-2:", A_gpt2.shape, tok_g)

### 6.1 Búsqueda automática de cabezas con patrones conocidos

Para cada cabeza de BERT se calcula la atención promedio al token anterior, al siguiente, a sí mismo, a `[CLS]` (primer token) y a `[SEP]` (último). Se reporta la cabeza más fuerte de cada patrón y, como resumen, cuántas cabezas dedican más del 50 % de su atención a un solo patrón.

In [ ]:
pats = head_patterns(A_bert)  # cada valor: (12, 12)
best = {}
for name, M in pats.items():
    idx = M.argmax().item(); l, h = divmod(idx, 12)
    best[name] = (l, h, M[l, h].item())
    print(f"{name:>9}: capa {l:2d}, cabeza {h:2d} → {M[l, h]:.2f} de atención promedio; "
          f"cabezas con >50 %: {(M > 0.5).sum().item()}")

# atención a puntuación ("," y ".") en BERT
punct = [i for i, t in enumerate(tok_b) if t in {",", "."}]
P = A_bert[..., :, punct].sum(-1).mean(-1)
l, h = divmod(P.argmax().item(), 12)
best["puntuación"] = (l, h, P[l, h].item())
print(f"puntuación: capa {l}, cabeza {h} → {P[l, h]:.2f}")

In [ ]:
sel = [("anterior", "token anterior"), ("siguiente", "token siguiente"),
       ("primero", "[CLS]"), ("último", "[SEP]")]
heads_bert = [(best[k][0], best[k][1], f"{lbl} ({best[k][2]:.2f})") for k, lbl in sel]
fig = plot_heads(A_bert, tok_b, heads_bert, title="BERT: cabezas especializadas")
fig.savefig(FIG / "bert_cabezas.png"); plt.show()

pats_g = head_patterns(A_gpt2)
l_prev, h_prev = divmod(pats_g["anterior"].argmax().item(), 12)
l_self, h_self = divmod(pats_g["misma"][1:].argmax().item() + 12, 12)  # excluye capa 0
heads_gpt2 = [(0, 0, "capa inicial"), (l_prev, h_prev, "token anterior"),
              (l_self, h_self, "misma palabra"), (11, 0, "capa final")]
fig = plot_heads(A_gpt2, tok_g, heads_gpt2, title="GPT-2: patrones causales")
fig.savefig(FIG / "gpt2_cabezas.png"); plt.show()

**Patrones identificados.** En BERT aparecen cabezas casi deterministas: una diagonal desplazada hacia el **token siguiente** en capas bajas (capa 2), otra hacia el **token anterior** (capa 3), una que manda casi todo a **[CLS]** (capa 1) y muchas en capas medias (5–9) que concentran la mayor parte de su atención en **[SEP]**. Clark et al. (2019) interpretan esa atención a [SEP] como una operación «nula» (*no-op*): cuando la cabeza no encuentra nada relevante, deposita su masa en un token neutro. También hay cabezas que atienden a la puntuación (coma y punto) y a la propia palabra. En GPT-2 la máscara causal deja todo por debajo de la diagonal; hay una cabeza de **token anterior** muy marcada y, en las capas profundas, la primera columna (token «The») acapara la atención: es el *attention sink* que se analiza en la sección 8.

## 7. Correferencia: «it» → *animal* o *street*

Se buscan cabezas de BERT en las que el token «it» atienda más a «animal» en la oración con *tired* y más a «street» en la oración con *wide*. Se ordenan las cabezas por $\Delta = [a(\text{animal})-a(\text{street})]_{tired} - [a(\text{animal})-a(\text{street})]_{wide}$.

In [ ]:
s_tired = "The animal didn't cross the street because it was too tired."
s_wide = "The animal didn't cross the street because it was too wide."
tok_t, A_t = get_attentions(bert, tok_bert, s_tired)
tok_w, A_w = get_attentions(bert, tok_bert, s_wide)
i_it, i_an, i_st = tok_t.index("it"), tok_t.index("animal"), tok_t.index("street")

d_t = A_t[:, :, i_it, i_an] - A_t[:, :, i_it, i_st]
d_w = A_w[:, :, i_it, i_an] - A_w[:, :, i_it, i_st]
flip = (d_t > 0) & (d_w < 0)  # animal gana con "tired" y street gana con "wide"
print(f"Cabezas que cambian de referente: {flip.sum().item()} de 144")
order = torch.argsort((d_t - d_w).flatten(), descending=True)
print(f"{'cabeza':>8} | tired: animal  street | wide: animal  street")
for i in order[:6]:
    l, h = divmod(i.item(), 12)
    mark = "  ← cambia" if flip[l, h] else ""
    print(f"  L{l:>2}H{h:<2} |        {A_t[l,h,i_it,i_an]:.3f}   {A_t[l,h,i_it,i_st]:.3f} |       "
          f"{A_w[l,h,i_it,i_an]:.3f}   {A_w[l,h,i_it,i_st]:.3f}{mark}")

In [ ]:
# Cabeza con el cambio más claro entre las que cambian de referente
L_it, H_it = divmod(torch.where(flip, d_t - d_w, torch.tensor(-1e9)).argmax().item(), 12)
fig, axes = plt.subplots(1, 2, figsize=(10, 2.6), sharey=True)
for ax, toks, A, name in [(axes[0], tok_t, A_t, "too tired"), (axes[1], tok_w, A_w, "too wide")]:
    row = A[L_it, H_it, i_it].numpy()
    colors = ["tab:orange" if t in ("animal", "street") else "tab:blue" for t in toks]
    ax.bar(range(len(toks)), row, color=colors)
    ax.set_xticks(range(len(toks)), toks, rotation=60, fontsize=8)
    ax.set_title(f"«it» en «…{name}.» (capa {L_it}, cabeza {H_it})", fontsize=9)
axes[0].set_ylabel("peso de atención desde «it»")
fig.tight_layout(); fig.savefig(FIG / "it_animal_street.png"); plt.show()

**Resultado:** existen varias cabezas (principalmente en capas 6–11) en las que «it» se desplaza de *animal* a *street* al cambiar el adjetivo; la más clara es la capa 6, cabeza 10. No obstante, en muchas cabezas «it» atiende más a *animal* en ambas oraciones, o la mayor parte de su masa va a [SEP] o a la puntuación, así que la «resolución de correferencia» no está concentrada en una sola cabeza.

## 8. GPT-2: máscara causal y *attention sinks*

In [ ]:
upper = A_gpt2.triu(diagonal=1)
print("Máximo peso por encima de la diagonal (todas las capas/cabezas):", upper.abs().max().item())
print("¿Triangular inferior en las 144 cabezas?", bool((upper == 0).all()))
print("¿Filas suman 1?", torch.allclose(A_gpt2.sum(-1), torch.ones_like(A_gpt2.sum(-1)), atol=1e-5))

# Proporción de atención al primer token por capa (se excluye la fila 0, que solo puede verse a sí misma)
first_share = A_gpt2[:, :, 1:, 0].mean(dim=(1, 2))
uniform_share = torch.tensor([1 / (i + 1) for i in range(1, A_gpt2.shape[-1])]).mean()

# Control: misma medición en un texto más largo y con un primer token distinto
LONG = ("Yesterday my sister bought a new bicycle, and after lunch we rode it along the river "
        "until the sun went down behind the old bridge near the station.")
tok_long, A_long = get_attentions(gpt2, tok_gpt2, LONG)
first_long = A_long[:, :, 1:, 0].mean(dim=(1, 2))
for l in range(12):
    print(f"capa {l:2d}: primer token recibe {first_share[l]:.1%} (oración corta) | {first_long[l]:.1%} (oración larga)")
print(f"referencia si la atención fuera uniforme: {uniform_share:.1%}")

fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(range(12), first_share, "o-", label=f"«{tok_g[0]}…» (n={len(tok_g)})")
ax.plot(range(12), first_long, "s-", label=f"«{tok_long[0]}…» (n={len(tok_long)})")
ax.axhline(uniform_share, color="k", ls="--", lw=0.8, label="uniforme (oración corta)")
ax.set(xlabel="capa", ylabel="atención al primer token", title="GPT-2: attention sink en el primer token", ylim=(0, 1))
ax.legend(fontsize=8); fig.tight_layout(); fig.savefig(FIG / "gpt2_sink.png"); plt.show()

**Attention sinks (Xiao et al., 2023).** La softmax obliga a que cada fila sume 1, así que una cabeza que «no quiere» mirar a nada igual debe repartir su masa en algún sitio. En modelos causales, el primer token es visible desde todas las posiciones durante todo el entrenamiento, y el modelo aprende a usarlo como depósito de esa atención sobrante independientemente de su contenido. Se observa que la proporción hacia el primer token crece con la profundidad (de ~20 % en la capa 0 a ~70–80 % en capas medias/profundas), muy por encima de lo uniforme, y que ocurre también con otro primer token («Yesterday»). Xiao et al. muestran que si en inferencia con ventana deslizante se descartan esos primeros tokens la perplejidad explota; conservando unos pocos *sink tokens* (StreamingLLM) el modelo genera estable sobre millones de tokens.

## 9. Entropía de la atención por capa

Entropía de cada fila $H=-\sum_j a_{ij}\log a_{ij}$ promediada sobre cabezas y posiciones. Como en GPT-2 la fila $i$ solo ve $i+1$ tokens (entropía máxima $\log(i+1)$, menor que la $\log n$ de BERT), también se reporta la **entropía normalizada** $H/H_{max}\in[0,1]$ para comparar ambos modelos de forma justa.

In [ ]:
texts = [SENT, s_tired, s_wide, LONG,
         "Attention mechanisms let neural networks focus on the most relevant parts of the input."]
def layer_entropy(model, tok, normalize):
    vals = []
    for t in texts:
        _, A = get_attentions(model, tok, t)
        H = attention_entropy(A, normalize=normalize)
        if model is gpt2:
            H = H[..., 1:]  # la fila 0 de GPT-2 solo ve un token (entropía 0 trivial)
        vals.append(H.mean(dim=(1, 2)))
    return torch.stack(vals).mean(0)

ent = {(m, nrm): layer_entropy(model, tok, nrm)
       for m, model, tok in [("BERT", bert, tok_bert), ("GPT-2", gpt2, tok_gpt2)] for nrm in (False, True)}
print("capa | BERT H  GPT-2 H | BERT H/Hmax  GPT-2 H/Hmax")
for l in range(12):
    print(f"{l:4d} | {ent['BERT', False][l]:6.3f} {ent['GPT-2', False][l]:7.3f} | "
          f"{ent['BERT', True][l]:11.3f} {ent['GPT-2', True][l]:12.3f}")

fig, axes = plt.subplots(1, 2, figsize=(10, 3.2))
for m, mk in [("BERT", "o-"), ("GPT-2", "s-")]:
    axes[0].plot(range(12), ent[m, False], mk, label=m)
    axes[1].plot(range(12), ent[m, True], mk, label=m)
axes[0].set(title="Entropía media (nats)", xlabel="capa"); axes[1].set(title="Entropía normalizada H/Hmax", xlabel="capa", ylim=(0, 1))
for ax in axes: ax.legend()
fig.tight_layout(); fig.savefig(FIG / "entropia.png"); plt.show()

**Lectura:** en ambos modelos la capa 0 es la más difusa (mezcla amplia del contexto, cercana a un promedio). En BERT la entropía baja hasta las capas medias (5–7, donde dominan las cabezas que atienden a [SEP]) y vuelve a subir un poco en las últimas capas. En GPT-2 la atención se concentra cada vez más con la profundidad, en buena parte por el *attention sink*: si casi toda la masa va al primer token, la entropía es baja aunque la cabeza no esté «mirando» nada informativo. En conclusión, las capas profundas atienden de forma **más concentrada** que las primeras, y la baja entropía no implica necesariamente atención semánticamente selectiva.

## 10. (Opcional) bertviz

Visualización interactiva de las mismas oraciones; se renderiza solo dentro de Jupyter (no en el PDF).

In [ ]:
try:
    from bertviz import head_view
    enc = tok_bert(s_tired, return_tensors="pt")
    with torch.no_grad():
        att = bert(**enc).attentions
    head_view(att, tok_bert.convert_ids_to_tokens(enc["input_ids"][0]))
except ImportError:
    print("bertviz no está instalado: pip install bertviz")